# <font color='264CC7'> activity_tracker </font>

Este notebook tiene como objetivo desarrollar y validar progresivamente los módulos que conforman el sistema telegram-notion-activity-tracker. Cada sección corresponde a un componente funcional independiente, que será probado de manera aislada antes de integrarse en el flujo completo.

El sistema estará compuesto por los siguientes módulos:
- Clasificador: analiza el texto de la actividad y lo categoriza según la taxonomía definida.
- Transcriptor: convierte mensajes de voz en texto procesable.
- Envío a Notion: construye el registro estructurado y lo inserta en la base de datos.
- Bot de Telegram: recibe mensajes y audios, y activa el flujo de procesamiento.
- Pipeline: integra todos los módulos en un proceso completo desde la captura hasta el almacenamiento.

Los paquetes necesarios son:

In [8]:
# !pip install pymupdf 
# !pip install openai
# !pip install notion-client
# !pip install pyTelegramBotAPI

---
## <font color='264CC7'> Clasificador </font>


In [9]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
import json

In [10]:
# Configurar la clave de API
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [11]:
def load_categories(file_path: str) -> dict:
    with open(file_path, 'r', encoding='utf-8') as file:
        categories = json.load(file)
    return categories

categories = load_categories('categories.json')
print(categories)

{'proyectos': {'nombre': 'Diseño, seguimiento y evaluación de proyectos de carreras y programas', 'descripcion': 'Actividades relacionadas con la planificación, monitoreo y evaluación de proyectos académicos vinculados a carreras y programas de la Facultad.'}, 'internacionalizacion': {'nombre': 'Internacionalización curricular y movilidad docente', 'descripcion': 'Acciones orientadas a fortalecer la dimensión internacional del currículo y promover la movilidad académica docente.'}, 'recursos': {'nombre': 'Optimización de los recursos didácticos y de la información para el aprendizaje', 'descripcion': 'Gestión y mejora de materiales didácticos, repositorios académicos y recursos de información para apoyar el proceso de enseñanza-aprendizaje.'}, 'rda': {'nombre': 'Medición de los resultados de aprendizaje', 'descripcion': 'Diseño, aplicación y análisis de instrumentos y estrategias para evaluar el logro de los resultados de aprendizaje.'}, 'areas': {'nombre': 'Organización de áreas curri

In [12]:
class Activity(BaseModel):
    name: str
    description: str
    date: str = Field(description="Fecha en formato AAAA-MM-DD.")
    category: str

def classify_activity(texto: str, categories: dict, today: str) -> str:

    allowed = list(categories.keys())

    system_prompt = (
        "Eres un asistente para registrar actividades de gestión académica de la USDIC en la FCENA (PUCE). "
        "Contexto institucional: FCENA incluye las carreras de Biología, Microbiología, Química, Bioingeniería, "
        "Ciencias Biomédicas y Ciencia de Datos; y programas de posgrado de Maestría en Biología y "
        "Maestría en Ciencias Actuariales. "
        "Conceptos clave:\n"
        "- Rutas académicas: procesos para que un estudiante apruebe un resultados de aprendizaje (RdA) pendiente.\n"
        "- Banner: sistema de gestión académica utilizado para matrícula, registro de notas, programación académica.\n"
        "- EVA: Entorno Virtual de Aprendizaje (aulas virtuales).\n"
        "Tu tarea es estructurar y clasificar actividades relacionadas con estos ámbitos. "
        "No inventes información; si algo no está explícito, manténlo general."
    )
    user_prompt = (
        "1) Elige EXACTAMENTE una categoría de la lista permitida.\n"
        "2) Genera un nombre corto y simple pero que capture lo que se hace (3–8 palabras).\n"
        "3) Redacta una descripción breve (1 frases).\n"
        f"4) Si no hay fecha explícita, usa la de hoy que es {today} (formato AAAA-MM-DD).\n"
        "5) Devuelve solo el objeto del esquema.\n\n"
        f"Categorías permitidas: {allowed}\n\n"
        f"Actividad: {texto}"
    )
    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
            ],
        text_format=Activity, 
        temperature=0
    )
    return response.output_parsed

In [13]:
texto = "Ayer generé los archivos de horarios de las diferentes carreras de la facultad que fueron enviado a estudiantes y a la DDE."

actividad_clasificada = classify_activity(texto, categories, "2026-20-02")
print(actividad_clasificada)

name='Generación y envío de horarios' description='Se generaron los archivos de horarios de las carreras y se enviaron a estudiantes y a la DDE.' date='2026-02-19' category='programacion'


In [14]:
texto = "Solicione un inconveniente reportado por estudiantes en el que no se habían pasado las notas de los itinerarios al sistema Banner"

actividad_clasificada = classify_activity(texto, categories, "2026-20-02")
print(actividad_clasificada)

name='Revisión de notas en Banner' description='Se atendió un inconveniente reportado por estudiantes sobre la falta de registro de notas de itinerarios en el sistema Banner.' date='2026-02-20' category='rda'


---
## <font color='264CC7'> Transcriptor </font>

---
## <font color='264CC7'> Envío a Notion </font>

---
## <font color='264CC7'> Bot de Telegram </font>

---
## <font color='264CC7'> Pipeline </font>